# Unstructured to Structured Data

## Text to JSON to DataFrame

In [1]:
from ollama import chat
from ollama import ChatResponse

In [2]:
from pydantic import BaseModel, PastDate
from pydantic.config import ConfigDict
from datetime import date
from pydantic.dataclasses import dataclass

In [3]:
import pandas as pd
import json

In [4]:
# text example
mytext = '''
    FESTIVAL. When the Nordic region’s largest film festival, the Göteborg Film Festival, 
    opens on 23 January, Danish cinema will be strongly represented. 
    A total of six Danish feature films and two documentaries have been selected for this year’s festival programme. 
    In the main competition for Nordic feature films, 
    Maria Sødahl’s ‘The Last Resort’ and Emilie Thalund’s feature debut ‘Weightless’ will compete 
    for the award for Best Nordic Film and a prize of SEK 400,000. 
'''

In [5]:
prompt = f'''
        You are part of movie industry news-reporting system, extract key values 
        from the below listed events: {mytext}
'''

In [6]:
# we design the structure as we need it
class MyModel(BaseModel):
    festival: str
    event_date: str 
    feature_films_title: list[str]
    award: int
    currency: str

    model_config = ConfigDict(title='MyModel')

In [7]:
# use LLM to help building the structure
response = chat(
    model='llama3.2:latest', 
    messages=[
      {
        'role': 'user',
        'content': mytext,
      },
    ], format = MyModel.model_json_schema())

In [8]:
# print(response['message']['content'])
response.message.content

'{\n    "festival": "Göteborg Film Festival",\n    "event_date": "23 January",\n    "feature_films_title": [\n        "The Last Resort",\n        "Weightless"\n    ],\n    "award": 400000,\n    "currency": "SEK"\n}'

In [9]:
validated = MyModel.model_validate_json(response.message.content)
schema = MyModel.model_json_schema(response.message.content)

In [10]:
schema

{'properties': {'festival': {'title': 'Festival', 'type': 'string'},
  'event_date': {'title': 'Event Date', 'type': 'string'},
  'feature_films_title': {'items': {'type': 'string'},
   'title': 'Feature Films Title',
   'type': 'array'},
  'award': {'title': 'Award', 'type': 'integer'},
  'currency': {'title': 'Currency', 'type': 'string'}},
 'required': ['festival',
  'event_date',
  'feature_films_title',
  'award',
  'currency'],
 'title': 'MyModel',
 'type': 'object'}

In [11]:
json_data = dict(validated)
json_data

{'festival': 'Göteborg Film Festival',
 'event_date': '23 January',
 'feature_films_title': ['The Last Resort', 'Weightless'],
 'award': 400000,
 'currency': 'SEK'}

In [12]:
# print(json.dumps(schema, indent=2))

In [13]:
df = pd.DataFrame.from_dict(json_data)

In [14]:
df

,festival,event_date,feature_films_title,award,currency
0,Göteborg Film Festival,23 January,The Last Resort,400000,SEK
1,Göteborg Film Festival,23 January,Weightless,400000,SEK
